# The Machine: A Bits and Binary Playground

Everything a computer does — every photo, song, spreadsheet, and line of code — is stored as **bits**: switches that are either off (`0`) or on (`1`). That sounds impossibly crude, and yet it is enough to represent numbers, negative numbers, text in every human language, and (almost) fractions. In this lab you will poke at all of those representations with real Python code.

**How to use this notebook:** click a code cell and press `Shift+Enter` to run it. Run the cells from top to bottom — later cells sometimes reuse functions defined in earlier ones. If things ever get confused, use *Kernel → Restart Kernel and Run All Cells*.

Don't worry if some Python syntax (like `for` loops) is new to you. For now, treat each code cell as a small machine: run it, look at the output, and match it to the explanation around it.

## Counting in binary

We count in base 10 because we have ten fingers: the digits run 0–9, and each column is worth ten times the one to its right. Binary is base 2: the only digits are 0 and 1, and each column is worth *twice* the one to its right — ones, twos, fours, eights, and so on.

So binary `1101` means $1 \times 8 + 1 \times 4 + 0 \times 2 + 1 \times 1 = 13$.

Python's built-in `bin()` function shows any integer in binary. The `0b` prefix is just Python's way of saying "what follows is binary".

In [ ]:
# The numbers 0 to 10, in decimal and in binary
for n in range(11):
    print(f"{n:>3}  ->  {bin(n)}")

Read the output slowly and watch the pattern: the rightmost bit flips on every step (just like the ones digit in decimal), the next bit flips every *two* steps, the next every *four*. Notice how 8 is `0b1000` — a single 1 followed by zeros, exactly like 1000 in decimal is a power of ten.

Conversion works in both directions. `int(text, base)` reads a string of digits in any base you name:

In [ ]:
# From binary text back to a number
print(int("1101", 2))       # 8 + 4 + 0 + 1 = 13

# f-strings can format a number in binary (:b) with zero-padding (:08b)
n = 13
print(f"{n} in binary is {n:b}")
print(f"{n} padded to 8 bits is {n:08b}")

# Round trip: number -> binary text -> number
text = f"{202:b}"
print(text, "->", int(text, 2))

## Hexadecimal: binary for humans

Long strings of bits are hard to read, so programmers use **hexadecimal** (base 16, "hex" for short): digits 0–9 then letters a–f for the values ten to fifteen. The magic property is that *one hex digit is exactly four bits*, so hex is really just a compact spelling of binary. You will meet hex in colour codes (`#ff8800`), memory addresses, and error dumps.

In [ ]:
# One hex digit = four bits: the full table
print("dec | binary | hex")
print("----+--------+----")
for n in range(16):
    print(f" {n:>2} |  {n:04b}  |  {n:x}")

Check a row against the column headers: 10 is `1010` in binary and `a` in hex. Now a two-digit hex number is just two four-bit groups glued together — `0xff` is `1111 1111`, which is 255.

In [ ]:
print(hex(255), bin(255))          # 0xff is eight 1-bits
print(int("ff", 16))               # hex text -> number
print(int("ff8800", 16))           # the orange from a web colour code

# Split a colour code into its red, green, blue bytes
colour = 0xFF8800
print("red:", colour >> 16, " green:", (colour >> 8) & 0xFF, " blue:", colour & 0xFF)

## How much fits in $n$ bits?

Each extra bit doubles the number of patterns you can make: 1 bit gives 2 patterns, 2 bits give 4, and $n$ bits give $2^n$. This one formula explains a surprising number of computing facts — why a **byte** (8 bits) holds 256 values, why old game consoles were "16-bit", and why file sizes jump in powers of two.

In [ ]:
for bits in [1, 2, 4, 8, 16, 32]:
    print(f"{bits:>2} bits -> {2**bits:,} different patterns")

Eight bits give 256 patterns — enough for every character on a US keyboard with room to spare, which is why the byte became the standard unit. Thirty-two bits already give about 4.3 billion patterns; that is why a 32-bit counter of seconds runs for roughly 136 years before wrapping around.

## Negative numbers: two's complement

So far every pattern stood for a *non-negative* number. But the machine also needs $-7$. The trick used by essentially every modern CPU is called **two's complement**: with $n$ bits, a negative number $-x$ is stored as the ordinary bit pattern of $2^n - x$. In 8 bits, $-1$ becomes $256 - 1 = 255$, i.e. `11111111`.

Two consequences worth memorising:

- the leftmost bit acts as a *sign flag* — 1 means negative;
- 8 bits cover $-128$ to $+127$ (one extra value on the negative side).

Let's build an encoder.

In [ ]:
def to_twos(n, bits=8):
    # Encode integer n as a two's-complement bit string of the given width.
    lo, hi = -(2 ** (bits - 1)), 2 ** (bits - 1) - 1
    if not lo <= n <= hi:
        raise ValueError(f"{n} does not fit in {bits} bits ({lo}..{hi})")
    if n < 0:
        n = n + 2 ** bits        # the two's-complement wrap-around
    return f"{n:0{bits}b}"

for value in [0, 1, 7, 127, -1, -2, -7, -128]:
    print(f"{value:>5}  ->  {to_twos(value)}")

Look at `-1`: all ones. That is the classic two's-complement fingerprint. And notice how `7` starts with `0` while `-7` starts with `1` — the sign bit at work.

Decoding goes the other way: read the bits as an ordinary unsigned number, and if the sign bit is set, subtract $2^n$. Let's write the decoder and then *prove* the pair works by round-tripping every single 8-bit value.

In [ ]:
def from_twos(bit_string):
    # Decode a two's-complement bit string back into a Python integer.
    bits = len(bit_string)
    value = int(bit_string, 2)
    if value >= 2 ** (bits - 1):   # sign bit set -> negative number
        value -= 2 ** bits
    return value

print(from_twos("11111111"), from_twos("11111001"), from_twos("00000111"))

# Round-trip test: every value from -128 to 127 must survive encode+decode
for n in range(-128, 128):
    assert from_twos(to_twos(n)) == n
print("All 256 eight-bit values round-trip correctly.")

That `assert` line is a tiny automated test: it crashes loudly if the claim is ever false, and stays silent when all is well. Getting "All 256 values round-trip" means our encoder and decoder are exact inverses — not just plausible, *verified*.

## Text is numbers too: `ord` and `chr`

Computers store text by agreeing on a giant numbered list of characters. The modern list is **Unicode**: every character in (nearly) every writing system gets a number called its **code point**. Python gives you both directions: `ord(character)` returns the code point, `chr(number)` returns the character.

In [ ]:
print(ord("A"), ord("B"), ord("a"))
print(chr(65), chr(97), chr(63))

# A word is just a sequence of numbers wearing letters as a costume:
for ch in "Hello!":
    print(f"  {ch!r}  code point {ord(ch):>3}  hex {ord(ch):04x}  binary {ord(ch):08b}")

Two details worth spotting: capital `A` is 65 while lowercase `a` is 97 — a difference of exactly 32, one bit — and the code points for A–Z run consecutively, which is why sorting text sorts alphabetically (at least for plain English).

The first 128 code points are the old American **ASCII** table from the 1960s. Unicode extends it to over 150,000 characters. Code points above 127 are usually written `U+XXXX` in hexadecimal. To *store* them as bytes, Python (and most of the web) uses the **UTF-8** encoding, which cleverly spends between 1 and 4 bytes per character — cheap for English, and still able to reach every emoji.

In [ ]:
for ch in ["a", "Z", "é", "π", "€", "🐍"]:
    byte_len = len(ch.encode("utf-8"))
    print(f"  {ch}  U+{ord(ch):04X}  code point {ord(ch):>6,}  UTF-8 uses {byte_len} byte(s)")

The plain letters cost 1 byte, the accented `é` and Greek `π` cost 2, the euro sign 3, and the snake emoji a full 4 bytes. So "how many characters?" and "how many bytes?" are *different questions* — a distinction that bites real programs (think length limits in databases or text messages).

## The floating-point surprise

Now for the most famous gotcha in all of programming. Predict the output of the next cell before running it.

In [ ]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)

Not a Python bug — you'll get the same in Java, C, and JavaScript. Here's why: computers store fractions in *binary*, and $0.1$ in binary is $0.000110011001100...$ — an infinitely repeating pattern, just like $1/3 = 0.333...$ never terminates in decimal. The machine keeps only 64 bits (a format called **double-precision floating point**), so it stores a value *extremely close to* 0.1, but not exactly 0.1. Add two such near-misses and the error peeks out in the 17th digit.

Let's see the stored value with more digits than `print` normally shows:

In [ ]:
print(f"{0.1:.20f}")     # what the machine actually stores for 0.1
print(f"{0.25:.20f}")    # 0.25 is 1/4 = binary 0.01 -> stored EXACTLY

# The practical fixes:
import math
print(math.isclose(0.1 + 0.2, 0.3))          # compare with a tolerance
print(round(0.1 + 0.2, 9) == round(0.3, 9))  # or round before comparing

from decimal import Decimal                  # exact decimal arithmetic (slower)
print(Decimal("0.1") + Decimal("0.2"))

Three takeaways: powers of two like 0.25 are stored exactly; **never compare floats with `==`** — use `math.isclose`; and when money is involved, use `Decimal` (or count whole cents as integers), because "close enough" is not a policy banks accept.

## A `sys.getsizeof` safari

Last stop: how much memory do Python values actually occupy? `sys.getsizeof(x)` reports the size of an object in bytes. The numbers are bigger than you might guess, because a Python object is not just raw bits — it also carries bookkeeping (its type, reference counts, and so on).

In [ ]:
import sys

for value in [0, 1, 2**30, 2**100, 3.14, True, None, "", "a", "hello"]:
    print(f"{str(value)[:20]:>20}  ->  {sys.getsizeof(value)} bytes")

An honest fine-print note: the exact byte counts depend on the Python version and platform — in this browser notebook Python runs on WebAssembly, so your numbers may differ from a friend's laptop. The *shape* of the results is what matters:

- even the number `0` costs a few dozen bytes of object overhead — far more than the 4 or 8 bytes a raw machine integer needs;
- Python integers grow to hold *any* size (that `2**100` fits at all is remarkable — Java's `int` caps out at $2^{31}-1$), and bigger integers cost more bytes;
- each extra character in a plain string costs about one extra byte, on top of a fixed overhead.

Let's watch the string growth directly:

In [ ]:
import sys

previous = None
for length in [0, 1, 2, 4, 8, 16, 32, 64]:
    s = "x" * length
    size = sys.getsizeof(s)
    delta = "" if previous is None else f"  (+{size - previous} bytes)"
    print(f"length {length:>3}  ->  {size} bytes{delta}")
    previous = size

A fixed baseline plus roughly one byte per character — the overhead is the price of Python's convenience (strings that know their own length, work with any alphabet, and never overflow).

## What you just learned

- Binary and hex are positional number systems; $n$ bits give $2^n$ patterns, and one hex digit is four bits.
- Negative integers live as two's complement: $-x$ is stored as $2^n - x$, and the top bit signals the sign.
- Text is numbers: `ord`/`chr` map characters to Unicode code points, and UTF-8 spends 1–4 bytes per character.
- Binary fractions can't represent 0.1 exactly, so float arithmetic is *approximate* — compare with tolerance, never `==`.
- Python objects carry overhead you can measure with `sys.getsizeof`.

## Try it yourself

The exercises below have scaffolding that runs as-is. Replace the `# your code here` parts, then un-comment the test lines to check your work.

### Exercise 1 — Binary birthday

Convert the year you were born to binary using an f-string, and check your answer with `int(..., 2)`. Before running: how many bits do you *predict* a year around 2000 needs? (Hint: where does $2^{10} = 1024$ sit relative to it?)

In [ ]:
year = 2008            # <- change this to your own birth year
binary_year = ""       # your code here: format `year` in binary with an f-string

print("binary:", binary_year)
print("number of bits:", len(binary_year))
# Uncomment to verify the round trip once binary_year is filled in:
# assert int(binary_year, 2) == year

### Exercise 2 — `bits_needed`

Write a function `bits_needed(n)` that returns how many bits are required to write the positive integer `n` in binary. Two possible strategies: keep doubling a power of two until it exceeds `n` and count the doublings, or use `len(f"{n:b}")`.

In [ ]:
def bits_needed(n):
    # your code here
    pass

# Uncomment to test:
# assert bits_needed(1) == 1
# assert bits_needed(2) == 2
# assert bits_needed(255) == 8
# assert bits_needed(256) == 9
# print("bits_needed passes all tests!")

### Exercise 3 — Sixteen-bit range

Our `to_twos` function (still defined from earlier — this is one notebook-wide workspace) takes a `bits` argument. With 16 bits, what are the most negative and most positive values that fit? Work it out from the $2^{n-1}$ rule, fill them in, and confirm that `to_twos` accepts both — and that one step beyond each raises `ValueError`.

In [ ]:
smallest = 0    # your code here: the most negative 16-bit value
largest = 0     # your code here: the most positive 16-bit value

print(smallest, "->", to_twos(smallest, 16))
print(largest, " ->", to_twos(largest, 16))
# Uncomment to confirm the edges are really edges:
# to_twos(smallest - 1, 16)   # should raise ValueError

### Exercise 4 — Secret message decoder

The list below is a message written as Unicode code points. Loop over it, convert each number with `chr()`, and glue the characters together into one string.

In [ ]:
code_points = [80, 121, 116, 104, 111, 110, 32, 114, 111, 99, 107, 115, 33]

message = ""
# your code here: loop over code_points and add chr(cp) to message

print(message)

### Exercise 5 — Float detective

For each fraction in the list below, decide whether the machine stores it exactly or approximately. Rule of thumb: a fraction is exact in binary only when its denominator is a power of two. Check your predictions by printing each value to 20 decimal places.

In [ ]:
fractions = [1/2, 1/3, 1/4, 1/5, 3/8, 1/10]

# your code here: for each value, print it with an f-string like f"{value:.20f}"
# Which ones show a clean tail of zeros? Do they match your predictions?